# Practical Exercise – Nano Banana 🍌 (Gemini Image)  
**Goal:** programmatically generate, edit, and transform images using the Gemini API “Nano Banana” models.

This notebook covers the required tasks:

1. Text-to-image generation (prompting)  
2. Detailed prompt + improve with a reference image  
3. Step-by-step enhancement through prompt adjustments  
4. Combine multiple images into one coherent image (multi-reference input)  
5. Restore & colorize an old / black-and-white photo  
6. Design a visual marketing post using AI image generation  

> Models referenced in Google’s Gemini Image docs:  
> - **Nano Banana:** `gemini-2.5-flash-image` (fast)  
> - **Nano Banana Pro:** `gemini-3-pro-image-preview` (higher quality, better text rendering, up to 4K, many references)


## 0) Setup

### Get an API key
Create a Gemini API key and set it as an environment variable:

- **Linux/macOS**
```bash
export GEMINI_API_KEY="YOUR_KEY"
```

- **Windows (PowerShell)**
```powershell
setx GEMINI_API_KEY "YOUR_KEY"
```

Then restart your notebook kernel.

### Install dependencies
This notebook uses `requests`, `Pillow`, and optional `google-genai` SDK.


In [ ]:
# If running in a fresh environment, uncomment:
# %pip install -U requests pillow google-genai

import os, json, base64, pathlib
import requests
from PIL import Image
from io import BytesIO

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("Missing GEMINI_API_KEY env var. Set it and restart the kernel.")

OUTPUT_DIR = pathlib.Path("outputs")
INPUT_DIR = pathlib.Path("inputs")
OUTPUT_DIR.mkdir(exist_ok=True)
INPUT_DIR.mkdir(exist_ok=True)

print("✅ Ready. Put any reference photos in ./inputs and outputs will be saved to ./outputs")


## 1) Utility functions (REST call + image helpers)

We’ll use the **REST endpoint** so this notebook runs anywhere.

- Endpoint pattern:  
`https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent`

We request both **TEXT** and **IMAGE** outputs, then extract image bytes from the response.


In [ ]:
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/models"

def img_to_inline_part(image: Image.Image, mime_type: str = "image/png") -> dict:
    """Convert PIL image to Gemini inline_data part (base64)."""
    buf = BytesIO()
    fmt = "PNG" if mime_type == "image/png" else "JPEG"
    image.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return {"inline_data": {"mime_type": mime_type, "data": b64}}

def load_image(path: str) -> Image.Image:
    return Image.open(path).convert("RGBA")

def save_image_bytes(img_bytes: bytes, out_path: str) -> str:
    out_path = str(out_path)
    with open(out_path, "wb") as f:
        f.write(img_bytes)
    return out_path

def parse_gemini_image_response(resp_json: dict):
    """Returns (texts: list[str], images: list[bytes])"""
    texts, images = [], []
    candidates = resp_json.get("candidates", [])
    if not candidates:
        return texts, images
    parts = candidates[0].get("content", {}).get("parts", [])
    for p in parts:
        if "text" in p and p["text"]:
            texts.append(p["text"])
        inline = p.get("inlineData") or p.get("inline_data")
        if inline and inline.get("data"):
            images.append(base64.b64decode(inline["data"]))
    return texts, images

def gemini_generate_image(
    prompt: str,
    model: str = "gemini-3-pro-image-preview",
    reference_images: list[Image.Image] | None = None,
    aspect_ratio: str = "1:1",
    image_size: str | None = None,  # "1K", "2K", "4K"
    tools: list[dict] | None = None,
    seed: int | None = None,
):
    """Generate or edit images using Gemini Image models."""
    reference_images = reference_images or []
    parts = [{"text": prompt}] + [img_to_inline_part(img) for img in reference_images]

    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {
            "responseModalities": ["TEXT", "IMAGE"],
            "imageConfig": {"aspectRatio": aspect_ratio},
        },
    }
    if image_size:
        payload["generationConfig"]["imageConfig"]["imageSize"] = image_size
    if seed is not None:
        payload["generationConfig"]["seed"] = seed
    if tools:
        payload["tools"] = tools

    url = f"{BASE_URL}/{model}:generateContent"
    headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}

    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=180)
    if not r.ok:
        raise RuntimeError(f"Gemini API error {r.status_code}: {r.text[:1000]}")
    return r.json()

def show_pil(img: Image.Image, max_size=768):
    import matplotlib.pyplot as plt
    w, h = img.size
    scale = min(1.0, max_size / max(w, h))
    img2 = img.resize((int(w * scale), int(h * scale)))
    plt.figure(figsize=(6, 6))
    plt.imshow(img2)
    plt.axis("off")
    plt.show()


## Task 1 — Generate an image from a text description (prompting)

We’ll generate one image from a text prompt.  
Tip: write **a mini scene** (subject + setting + lighting + camera + style).


In [ ]:
prompt_1 = '''
A cinematic photo of a nano-sized banana astronaut standing on a glossy black rock
on the moon, Earth visible in the background, dramatic rim lighting, ultra-detailed,
shallow depth of field, 35mm lens, realistic materials.
'''.strip()

resp = gemini_generate_image(
    prompt=prompt_1,
    model="gemini-2.5-flash-image",   # Nano Banana (fast)
    aspect_ratio="1:1",
)

texts, images = parse_gemini_image_response(resp)
print("\n".join(texts) if texts else "(no text)")
print(f"Images returned: {len(images)}")

out_path = OUTPUT_DIR / "task1_text_to_image.png"
save_image_bytes(images[0], out_path)
print("Saved:", out_path)

img = Image.open(out_path)
show_pil(img)


## Task 2 — Detailed prompt, then improve using a reference image

1) Generate an image from a detailed description  
2) Upload a reference image (e.g., a logo, character, product photo) and ask the model to improve while preserving key elements.

**Instructions:**  
- Put a reference image in `./inputs/reference_1.png` (or change the filename below).


In [ ]:
# Step A: generate a baseline image (no references)
prompt_2a = '''
A clean studio product photo of a futuristic banana-shaped USB drive on a neutral gray background,
soft box lighting, sharp focus, subtle shadow, minimalistic, premium look.
'''.strip()

resp_2a = gemini_generate_image(
    prompt=prompt_2a,
    model="gemini-2.5-flash-image",
    aspect_ratio="4:3",
)
_, imgs_2a = parse_gemini_image_response(resp_2a)
base_path = OUTPUT_DIR / "task2a_baseline.png"
save_image_bytes(imgs_2a[0], base_path)
print("Saved baseline:", base_path)
show_pil(Image.open(base_path))

# Step B: improve using a reference image (e.g., your brand logo / style guide / product sketch)
ref_path = INPUT_DIR / "reference_1.png"
if not ref_path.exists():
    print(f"⚠️ Put a reference image at: {ref_path.resolve()}")
else:
    ref_img = load_image(str(ref_path))
    prompt_2b = '''
Improve the product image using the reference image as the brand style guide.
Keep a premium minimalistic studio look, preserve the banana-shaped USB drive concept,
enhance reflections and material realism, and make it look like a real photographed product.
'''.strip()

    resp_2b = gemini_generate_image(
        prompt=prompt_2b,
        model="gemini-3-pro-image-preview",  # Pro for higher quality
        reference_images=[ref_img],
        aspect_ratio="4:3",
        image_size="2K",
    )
    _, imgs_2b = parse_gemini_image_response(resp_2b)
    improved_path = OUTPUT_DIR / "task2b_improved_with_reference.png"
    save_image_bytes(imgs_2b[0], improved_path)
    print("Saved improved:", improved_path)
    show_pil(Image.open(improved_path))


## Task 3 — Generate an image and enhance it step-by-step through prompt adjustments

We’ll do iterative prompting: generate → critique → adjust prompt → regenerate.

You can change the `prompts` list to your own iterations.


In [ ]:
prompts = [
    # v1
    'A poster-style illustration of a cyberpunk banana in a rainy neon street, bold typography, dramatic composition.',
    # v2 (more specific)
    '''Poster-style illustration: a cyberpunk banana character wearing a translucent raincoat in a rainy neon street at night.
Add reflections on wet asphalt, neon signs in the background, high contrast, clean vector-like edges, cinematic framing.
Typography: big title "NANO BANANA" at top, subtitle "FAST. FUN. FUTURISTIC." at bottom, legible modern sans-serif.''',
    # v3 (improve text + layout + margins)
    '''Design a polished marketing poster.
Scene: cyberpunk banana mascot in a neon rainy street at night; reflections on wet asphalt; strong rim light.
Layout: clear margins; centered composition; no clutter.
Text: Title "NANO BANANA" (large, crisp, modern sans-serif) at the top; subtitle "FAST. FUN. FUTURISTIC." smaller at bottom.
Ensure text is perfectly legible and not distorted. Professional print-ready look.''',
]

for i, p in enumerate(prompts, start=1):
    resp_i = gemini_generate_image(
        prompt=p.strip(),
        model="gemini-3-pro-image-preview",
        aspect_ratio="4:5",
        image_size="2K",
        seed=1234,  # remove if you want variety
    )
    texts_i, imgs_i = parse_gemini_image_response(resp_i)
    if texts_i:
        print(f"\n--- Notes v{i} ---\n" + "\n".join(texts_i))

    out = OUTPUT_DIR / f"task3_iter_{i}.png"
    save_image_bytes(imgs_i[0], out)
    print("Saved:", out)
    show_pil(Image.open(out))


## Task 4 — Combine two or more images into a single coherent image (Multi-Reference Input)

Put 2–3 images in `./inputs/`:

- `ref_person.png` (optional)
- `ref_style.png` (optional)
- `ref_background.png` (optional)

Then ask the model to combine them coherently.


In [ ]:
paths = [
    INPUT_DIR / "ref_person.png",
    INPUT_DIR / "ref_style.png",
    INPUT_DIR / "ref_background.png",
]
refs = [load_image(str(p)) for p in paths if p.exists()]

if len(refs) < 2:
    print("⚠️ Add at least TWO reference images in ./inputs (ref_person.png, ref_style.png, ref_background.png).")
else:
    prompt_4 = '''
Create one coherent image by combining the references:
- Use the first image as the main subject identity (keep face/character consistent).
- Use the second image as the style reference (color palette, rendering style).
- If a third image is provided, use it as the environment/background.
Produce a realistic, well-composed image with natural lighting and clean details.
'''.strip()

    resp_4 = gemini_generate_image(
        prompt=prompt_4,
        model="gemini-3-pro-image-preview",
        reference_images=refs,
        aspect_ratio="16:9",
        image_size="2K",
    )
    _, imgs_4 = parse_gemini_image_response(resp_4)
    out4 = OUTPUT_DIR / "task4_multi_reference_combo.png"
    save_image_bytes(imgs_4[0], out4)
    print("Saved:", out4)
    show_pil(Image.open(out4))


## Task 5 — Restore and colorize an old / black-and-white photograph

Put your old photo at `./inputs/old_photo_bw.jpg` (or update the filename below).

We’ll ask the model to:
- remove scratches/noise
- recover details
- colorize naturally
- upscale if possible


In [ ]:
old_path = INPUT_DIR / "old_photo_bw.jpg"
if not old_path.exists():
    print(f"⚠️ Put a black-and-white (or damaged) photo at: {old_path.resolve()}")
else:
    old_img = Image.open(old_path).convert("RGB")

    prompt_5 = '''
Restore this old photo:
- Remove scratches, dust, and compression artifacts.
- Recover facial details and edges naturally (do not over-smooth).
- Colorize realistically with natural skin tones and plausible clothing colors.
- Preserve the original composition and identity; do NOT change faces.
- Output a high-resolution, sharp result.
'''.strip()

    resp_5 = gemini_generate_image(
        prompt=prompt_5,
        model="gemini-3-pro-image-preview",
        reference_images=[old_img],
        aspect_ratio="3:4",
        image_size="2K",
    )
    _, imgs_5 = parse_gemini_image_response(resp_5)
    out5 = OUTPUT_DIR / "task5_restored_colorized.png"
    save_image_bytes(imgs_5[0], out5)
    print("Saved:", out5)
    show_pil(Image.open(out5))


## Task 6 — Design a visual marketing post using AI image generation

We’ll generate a **social media post** (4:5 works great for IG).  
We’ll use Nano Banana Pro for better text rendering.


In [ ]:
prompt_6 = '''
Create a modern Instagram marketing post (4:5) for an AI image tool called "Nano Banana".
Style: clean, premium, high-contrast, minimal.
Main visual: a playful banana icon subtly integrated into a sleek UI card mockup.
Text must be crisp and perfectly legible:
Headline: "CREATE IMAGES FAST"
Subhead: "Text • Edit • Multi-Reference • Restore"
CTA button text: "TRY IT NOW"
Use a modern sans-serif font, consistent spacing, strong hierarchy, and plenty of white space.
'''.strip()

resp_6 = gemini_generate_image(
    prompt=prompt_6,
    model="gemini-3-pro-image-preview",
    aspect_ratio="4:5",
    image_size="2K",
)

texts6, imgs6 = parse_gemini_image_response(resp_6)
if texts6:
    print("\n".join(texts6))

out6 = OUTPUT_DIR / "task6_marketing_post.png"
save_image_bytes(imgs6[0], out6)
print("Saved:", out6)
show_pil(Image.open(out6))


## Portfolio export

This cell zips all generated outputs into a single file you can submit as your portfolio.


In [ ]:
import zipfile, glob

zip_path = "nano_banana_portfolio.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob("outputs/*")):
        z.write(f)
print("✅ Portfolio zipped to:", zip_path)


## Short analysis & conclusions (what you should write in your submission)

For each task, include:

- **Prompt used** and why it was structured that way  
- **What changed** between iterations (Task 3) and the effect  
- **Which model** you used (Flash vs Pro) and why  
- **What you learned** about:
  - specificity (camera/lighting/composition)
  - reference-image roles (identity vs style vs background)
  - text rendering (Pro is usually better)
  - restoration (avoid “over-smoothing” and identity drift)

Suggested 5–8 bullets total. Keep it concise.
